# Agent Evaluation Notebook

This notebook evaluates the LangChain agent across four dimensions:

1. **Tool call accuracy**
2. **Response quality**
3. **Multi-step reasoning**
4. **Edge case handling**

**Prerequisites:**
- PostgreSQL running (`docker compose -f docker-compose.dev.yml up -d`)
- ChromaDB populated (`python scripts/embed_ingredients.py`)
- `OPENAI_API_KEY` set in `backend/.env`
- LangChain dependencies installed (`pip install langchain langchain-openai langgraph`)

## 1. Setup

In [17]:
import importlib
import app.services.agent
importlib.reload(app.services.agent)
from app.services.agent import create_chat_agent

In [18]:
import sys
import json
import time
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from app.core.config import OPENAI_API_KEY, AGENT_MODEL
from app.db.session import SessionLocal
from app.models.user import User
from app.models.product import Product
from app.models.routine import SkinProfile
from app.services.agent import create_chat_agent

print(f"OpenAI key set: {bool(OPENAI_API_KEY)}")
print(f"Agent model: {AGENT_MODEL}")

OpenAI key set: True
Agent model: gpt-4o-mini


In [19]:
db = SessionLocal()

# Create or reuse a test user
test_email = "agent_eval_test@shelflove.app"
test_user = db.query(User).filter(User.email == test_email).first()
if not test_user:
    from app.core.security import hash_password
    test_user = User(name="Eval Test", email=test_email, hashed_password=hash_password("test123"))
    db.add(test_user)
    db.commit()
    db.refresh(test_user)
    print(f"Created test user: id={test_user.id}")
else:
    print(f"Reusing test user: id={test_user.id}")

Reusing test user: id=2


In [20]:
# Seed test products for the evaluation
existing_products = db.query(Product).filter(Product.user_id == test_user.id).all()
if len(existing_products) < 3:
    products_to_add = [
        Product(user_id=test_user.id, name="CeraVe Moisturizing Cream", brand="CeraVe", category="skincare", product_type="moisturizer"),
        Product(user_id=test_user.id, name="The Ordinary Niacinamide 10%", brand="The Ordinary", category="skincare", product_type="serum"),
        Product(user_id=test_user.id, name="La Roche-Posay Anthelios SPF 50", brand="La Roche-Posay", category="skincare", product_type="spf"),
        Product(user_id=test_user.id, name="Cetaphil Gentle Cleanser", brand="Cetaphil", category="skincare", product_type="cleanser"),
        Product(user_id=test_user.id, name="Maybelline Fit Me Foundation", brand="Maybelline", category="makeup", product_type="other"),
    ]
    for p in products_to_add:
        db.add(p)
    db.commit()
    print(f"Added {len(products_to_add)} test products")
else:
    print(f"Reusing {len(existing_products)} existing products")

# Seed skin profile
profile = db.query(SkinProfile).filter(SkinProfile.user_id == test_user.id).first()
if not profile:
    profile = SkinProfile(
        user_id=test_user.id,
        skin_type="combination",
        is_sensitive=False,
        concerns=["acne", "hyperpigmentation"],
        goals=["clear skin", "even tone"],
    )
    db.add(profile)
    db.commit()
    print("Created skin profile: combination, acne, hyperpigmentation")
else:
    print(f"Reusing skin profile: {profile.skin_type}")

Reusing 5 existing products
Reusing skin profile: combination


## 2. Test queries (20 total)

In [21]:
test_queries = [
    # --- Category A: Ingredient lookups (5) ---
    {
        "id": 1,
        "category": "ingredient_lookup",
        "query": "Is retinol safe?",
        "expected_tools": ["lookup_ingredient_safety"],
        "notes": "Single ingredient, safety question",
    },
    {
        "id": 2,
        "category": "ingredient_lookup",
        "query": "What are the risks of salicylic acid?",
        "expected_tools": ["lookup_ingredient_safety"],
        "notes": "Risk-focused query",
    },
    {
        "id": 3,
        "category": "ingredient_lookup",
        "query": "Analyze these ingredients: water, glycerin, niacinamide, retinol",
        "expected_tools": ["lookup_ingredient_safety"],
        "notes": "Multi-ingredient list",
    },
    {
        "id": 4,
        "category": "ingredient_lookup",
        "query": "Is benzoyl peroxide good for oily skin?",
        "expected_tools": ["lookup_ingredient_safety"],
        "notes": "Skin-type context",
    },
    {
        "id": 5,
        "category": "ingredient_lookup",
        "query": "What's the comedogenicity of cetearyl alcohol?",
        "expected_tools": ["lookup_ingredient_safety"],
        "notes": "Specific property query",
    },

    # --- Category B: Product shelf queries (4) ---
    {
        "id": 6,
        "category": "product_query",
        "query": "What products do I have?",
        "expected_tools": ["query_user_products"],
        "notes": "No filter, all products",
    },
    {
        "id": 7,
        "category": "product_query",
        "query": "Show me my moisturizers",
        "expected_tools": ["query_user_products"],
        "notes": "Filter by product_type",
    },
    {
        "id": 8,
        "category": "product_query",
        "query": "Do I have any SPF products?",
        "expected_tools": ["query_user_products"],
        "notes": "Filter by product_type",
    },
    {
        "id": 9,
        "category": "product_query",
        "query": "What skincare products are in my shelf?",
        "expected_tools": ["query_user_products"],
        "notes": "Filter by category",
    },

    # --- Category C: Combined queries (4) ---
    {
        "id": 10,
        "category": "combined",
        "query": "Analyze the ingredients in my moisturizer",
        "expected_tools": ["query_user_products", "lookup_ingredient_safety"],
        "notes": "Two-step: find product then analyze ingredients",
    },
    {
        "id": 11,
        "category": "combined",
        "query": "Is the retinol in my serum safe for sensitive skin?",
        "expected_tools": ["query_user_products", "lookup_ingredient_safety"],
        "notes": "Product + ingredient + skin context",
    },
    {
        "id": 12,
        "category": "combined",
        "query": "Compare the safety of my cleanser and my moisturizer",
        "expected_tools": ["query_user_products", "lookup_ingredient_safety"],
        "notes": "Multi-product comparison",
    },
    {
        "id": 13,
        "category": "combined",
        "query": "What's the safest product in my shelf?",
        "expected_tools": ["query_user_products", "lookup_ingredient_safety"],
        "notes": "Broad analysis across all products",
    },

    # --- Category D: Summary requests (3) ---
    {
        "id": 14,
        "category": "summary",
        "query": "Give me a safety summary of niacinamide",
        "expected_tools": ["lookup_ingredient_safety", "summarize_safety"],
        "notes": "Lookup then summarize",
    },
    {
        "id": 15,
        "category": "summary",
        "query": "Is my retinol product safe? Summarize it for me",
        "expected_tools": ["query_user_products", "lookup_ingredient_safety", "summarize_safety"],
        "notes": "Three-step chain",
    },
    {
        "id": 16,
        "category": "summary",
        "query": "Summarize the risks of this ingredient list: water, fragrance, methylparaben",
        "expected_tools": ["lookup_ingredient_safety", "summarize_safety"],
        "notes": "Risk-focused summary",
    },

    # --- Category E: Edge cases (4) ---
    {
        "id": 17,
        "category": "edge_case",
        "query": "Hello!",
        "expected_tools": [],
        "notes": "Conversational, no tool needed",
    },
    {
        "id": 18,
        "category": "edge_case",
        "query": "What's the weather today?",
        "expected_tools": [],
        "notes": "Out of scope, should deflect",
    },
    {
        "id": 19,
        "category": "edge_case",
        "query": "asdfghjkl",
        "expected_tools": [],
        "notes": "Nonsense input",
    },
    {
        "id": 20,
        "category": "edge_case",
        "query": "Tell me about retinol and also what products I have and summarize everything",
        "expected_tools": ["lookup_ingredient_safety", "query_user_products", "summarize_safety"],
        "notes": "Complex multi-intent",
    },
]

print(f"{len(test_queries)} test queries defined")
for cat in ["ingredient_lookup", "product_query", "combined", "summary", "edge_case"]:
    count = sum(1 for q in test_queries if q["category"] == cat)
    print(f"  {cat:25s}  {count} queries")

20 test queries defined
  ingredient_lookup          5 queries
  product_query              4 queries
  combined                   4 queries
  summary                    3 queries
  edge_case                  4 queries


## 3. Run evaluation

In [22]:
def run_agent_query(agent, query: str) -> dict:
    """Invoke the agent and capture tool calls + response."""
    start = time.time()
    try:
        result = agent.invoke({"messages": [HumanMessage(content=query)]})
        elapsed = time.time() - start

        messages = result.get("messages", [])
        tool_calls_used = []
        tool_results = []
        final_response = ""

        for msg in messages:
            if isinstance(msg, AIMessage) and msg.tool_calls:
                for tc in msg.tool_calls:
                    tool_calls_used.append(tc["name"])
            elif isinstance(msg, AIMessage) and msg.content and not msg.tool_calls:
                final_response = msg.content
            elif isinstance(msg, ToolMessage):
                tool_results.append(msg.content[:200])  # truncate for display

        return {
            "success": True,
            "tool_calls": tool_calls_used,
            "tool_results": tool_results,
            "response": final_response,
            "latency": round(elapsed, 2),
            "error": None,
        }
    except Exception as e:
        elapsed = time.time() - start
        return {
            "success": False,
            "tool_calls": [],
            "tool_results": [],
            "response": "",
            "latency": round(elapsed, 2),
            "error": str(e),
        }

In [23]:
# Create a fresh agent for evaluation
agent = create_chat_agent(db, test_user.id)
print("Agent created successfully")

Agent created successfully


In [24]:
results = []

for tq in test_queries:
    print(f"[{tq['id']:2d}/20] {tq['query'][:60]:60s} ... ", end="", flush=True)
    result = run_agent_query(agent, tq["query"])
    result["query"] = tq
    results.append(result)
    status = "OK" if result["success"] else "FAIL"
    tools = result["tool_calls"]
    print(f"{status}  tools={tools}  {result['latency']}s")

print(f"\nCompleted: {sum(1 for r in results if r['success'])}/{len(results)} successful")

[ 1/20] Is retinol safe?                                             ... OK  tools=['lookup_ingredient_safety', 'summarize_safety']  7.29s
[ 2/20] What are the risks of salicylic acid?                        ... OK  tools=['lookup_ingredient_safety', 'summarize_safety']  6.1s
[ 3/20] Analyze these ingredients: water, glycerin, niacinamide, ret ... OK  tools=['lookup_ingredient_safety']  2.29s
[ 4/20] Is benzoyl peroxide good for oily skin?                      ... OK  tools=[]  1.18s
[ 5/20] What's the comedogenicity of cetearyl alcohol?               ... OK  tools=['lookup_ingredient_safety', 'summarize_safety']  5.48s
[ 6/20] What products do I have?                                     ... OK  tools=['query_user_products']  1.84s
[ 7/20] Show me my moisturizers                                      ... OK  tools=['query_user_products']  1.51s
[ 8/20] Do I have any SPF products?                                  ... OK  tools=['query_user_products']  1.2s
[ 9/20] What skincare products 

## 4. Tool call accuracy

In [25]:
def check_tool_accuracy(expected: list, actual: list) -> bool:
    """Check if the expected tools were called (order-insensitive, superset allowed)."""
    if not expected:
        return len(actual) == 0
    return set(expected).issubset(set(actual))


accuracy_rows = []
for r in results:
    tq = r["query"]
    correct = check_tool_accuracy(tq["expected_tools"], r["tool_calls"])
    accuracy_rows.append({
        "id": tq["id"],
        "category": tq["category"],
        "query": tq["query"][:50],
        "expected": tq["expected_tools"],
        "actual": r["tool_calls"],
        "correct": correct,
    })

print(f"{'ID':>3} {'Category':20s} {'Expected':40s} {'Actual':40s} {'Pass'}")
print("-" * 115)
for row in accuracy_rows:
    exp = str(row["expected"])
    act = str(row["actual"])
    mark = "PASS" if row["correct"] else "FAIL"
    print(f"{row['id']:3d} {row['category']:20s} {exp:40s} {act:40s} {mark}")

total = len(accuracy_rows)
passed = sum(1 for r in accuracy_rows if r["correct"])
print(f"\nTool call accuracy: {passed}/{total} ({passed/total*100:.0f}%)")

 ID Category             Expected                                 Actual                                   Pass
-------------------------------------------------------------------------------------------------------------------
  1 ingredient_lookup    ['lookup_ingredient_safety']             ['lookup_ingredient_safety', 'summarize_safety'] PASS
  2 ingredient_lookup    ['lookup_ingredient_safety']             ['lookup_ingredient_safety', 'summarize_safety'] PASS
  3 ingredient_lookup    ['lookup_ingredient_safety']             ['lookup_ingredient_safety']             PASS
  4 ingredient_lookup    ['lookup_ingredient_safety']             []                                       FAIL
  5 ingredient_lookup    ['lookup_ingredient_safety']             ['lookup_ingredient_safety', 'summarize_safety'] PASS
  6 product_query        ['query_user_products']                  ['query_user_products']                  PASS
  7 product_query        ['query_user_products']                  ['query_us

## 5. Response quality (manual scoring)

Run this cell to display responses for manual review. Score each on a 1-5 scale.

| Score | Meaning |
|-------|---------|
| 1 | Wrong, harmful, or completely unhelpful |
| 2 | Partially relevant but major issues |
| 3 | Acceptable, answers the question adequately |
| 4 | Good, accurate and well-formatted |
| 5 | Excellent, thorough and insightful |

In [26]:
for r in results:
    tq = r["query"]
    print(f"\n{'='*80}")
    print(f"Query {tq['id']} [{tq['category']}]: {tq['query']}")
    print(f"Tools called: {r['tool_calls']}")
    print(f"Latency: {r['latency']}s")
    print(f"\nResponse:\n{r['response']}")
    if r['error']:
        print(f"\nERROR: {r['error']}")

print(f"\n{'='*80}")
print("Review the responses above and fill in the quality_scores dict in the next cell.")


Query 1 [ingredient_lookup]: Is retinol safe?
Tools called: ['lookup_ingredient_safety', 'summarize_safety']
Latency: 7.29s

Response:
Retinol is generally safe for use, especially for your combination skin, but it can cause some side effects during the initial adjustment period. Here are the key points:

- **Safety Assessment**: Safety score of 5/10; it can cause peeling and dryness, particularly when you first start using it.
- **Benefits**: Effective for anti-aging, acne treatment, promoting cell turnover, stimulating collagen production, and reducing hyperpigmentation.
- **Risks**: Increased photosensitivity (always use SPF), potential to worsen rosacea, and it should be avoided during pregnancy due to teratogenic effects.

Make sure to monitor how your skin reacts and consider starting with a lower concentration if you're new to retinol!

Query 2 [ingredient_lookup]: What are the risks of salicylic acid?
Tools called: ['lookup_ingredient_safety', 'summarize_safety']
Latency: 6.1s

In [27]:
# Fill in scores after manual review (1-5 scale)
quality_scores = {
    # (helpfulness, accuracy, safety, conciseness)
    1:  (5, 5, 5, 4),  # retinol safety - excellent, personalized
    2:  (5, 5, 5, 4),  # salicylic acid risks - thorough, skin-type aware
    3:  (3, 2, 4, 4),  # multi-ingredient analyze - FAILED: said no data for ingredients that ARE in DB
    4:  (4, 2, 3, 4),  # benzoyl peroxide - FAILED: no tools called, violated "always look up" rule
    5:  (5, 5, 4, 5),  # cetearyl alcohol comedogenicity - directly answered
    6:  (5, 5, 5, 5),  # all products - perfect listing
    7:  (5, 5, 5, 5),  # moisturizers - correct filter
    8:  (4, 1, 4, 4),  # SPF products - WRONG: said none, but La Roche-Posay SPF 50 exists
    9:  (5, 5, 5, 5),  # skincare products - correct, noted makeup separately
    10: (3, 2, 4, 4),  # analyze moisturizer ingredients - FAILED: said no data
    11: (5, 5, 5, 4),  # retinol in serum safe? - excellent multi-tool chain
    12: (3, 2, 4, 4),  # compare cleanser vs moisturizer - FAILED: said no data for either
    13: (3, 2, 3, 4),  # safest product - FAILED: no actual comparison, fell back to general knowledge
    14: (5, 5, 5, 5),  # niacinamide summary - excellent
    15: (3, 4, 3, 4),  # retinol product summary - correct that no retinol product exists, but didn't look up retinol generically
    16: (5, 5, 5, 5),  # risk summary ingredient list - well-structured
    17: (5, 5, 5, 5),  # hello - appropriate
    18: (4, 5, 5, 5),  # weather - perfect deflection
    19: (4, 5, 5, 5),  # nonsense - polite handling
    20: (5, 5, 5, 5),  # multi-intent - handled all three brilliantly
}

if quality_scores:
    all_scores = list(quality_scores.values())
    avg_h = sum(s[0] for s in all_scores) / len(all_scores)
    avg_a = sum(s[1] for s in all_scores) / len(all_scores)
    avg_s = sum(s[2] for s in all_scores) / len(all_scores)
    avg_c = sum(s[3] for s in all_scores) / len(all_scores)
    print(f"Scored {len(quality_scores)}/{len(results)} queries")
    print(f"  Avg helpfulness:  {avg_h:.1f}/5")
    print(f"  Avg accuracy:     {avg_a:.1f}/5")
    print(f"  Avg safety:       {avg_s:.1f}/5")
    print(f"  Avg conciseness:  {avg_c:.1f}/5")
else:
    print("No scores entered yet. Fill in quality_scores after reviewing responses.")

Scored 20/20 queries
  Avg helpfulness:  4.3/5
  Avg accuracy:     4.0/5
  Avg safety:       4.5/5
  Avg conciseness:  4.5/5


## 6. Failure modes

Document failures and issues found during evaluation.

In [28]:
# Document failure modes here
failure_modes = [
    {
        "query_id": 3,
        "issue": "Multi-ingredient lookup returns 'no data found' for ingredients that exist in ChromaDB",
        "root_cause": "Agent passes comma-separated list to lookup_ingredient_safety, but the tool's retrieve_safety_records() may not be matching correctly through the agent path. Tool was called but results came back empty.",
        "fix": "Debug the lookup_ingredient_safety tool with multi-ingredient input. Check if normalize_ingredients() is splitting correctly when called through the agent vs directly.",
    },
    {
        "query_id": 4,
        "issue": "Agent did not call any tools for 'Is benzoyl peroxide good for oily skin?' — violated the system prompt rule to always look up ingredients first",
        "root_cause": "Agent decided it could answer from general knowledge without using the database. System prompt rule was not strong enough.",
        "fix": "Strengthen system prompt: 'NEVER answer ingredient safety questions without calling lookup_ingredient_safety first, even if you think you know the answer.'",
    },
    {
        "query_id": 8,
        "issue": "Agent reported no SPF products, but La Roche-Posay Anthelios SPF 50 (product_type='spf') exists in the test data",
        "root_cause": "Agent likely passed 'sunscreen' or 'spf product' as the product_type filter instead of the exact value 'spf'. Tool does exact string match.",
        "fix": "Update tool description to list the valid product_type values (cleanser, toner, serum, moisturizer, exfoliant, mask, spot_treatment, spf, oil, other) so the agent knows what to filter by.",
    },
    {
        "query_id": 10,
        "issue": "Combined query 'Analyze the ingredients in my moisturizer' — agent found the product but couldn't look up its ingredients",
        "root_cause": "Products table doesn't store ingredient lists. Agent tried to extract ingredients from the product name/brand, which doesn't work. There's no link between products and their ingredient text in the DB.",
        "fix": "Update system prompt to clarify: 'Product records do not contain ingredient lists. If a user asks to analyze a product's ingredients, ask them to provide the ingredient list from the product label, or ask them to scan the product first.'",
    },
    {
        "query_id": 15,
        "issue": "Agent didn't call lookup_ingredient_safety or summarize_safety — only called query_user_products twice",
        "root_cause": "User asked about 'my retinol product' and agent found no retinol product, so it stopped. Should have fallen back to looking up retinol safety data generically.",
        "fix": "Add to system prompt: 'If the user asks about a specific ingredient but doesn't have a matching product, still look up the ingredient safety data and provide general information.'",
    },
]

if failure_modes:
    print(f"Documented {len(failure_modes)} failure modes:")
    for fm in failure_modes:
        print(f"\n  Query {fm['query_id']}: {fm['issue']}")
        print(f"    Root cause: {fm['root_cause']}")
        print(f"    Fix: {fm['fix']}")
else:
    print("No failure modes documented yet. Fill in after reviewing results.")

Documented 5 failure modes:

  Query 3: Multi-ingredient lookup returns 'no data found' for ingredients that exist in ChromaDB
    Root cause: Agent passes comma-separated list to lookup_ingredient_safety, but the tool's retrieve_safety_records() may not be matching correctly through the agent path. Tool was called but results came back empty.
    Fix: Debug the lookup_ingredient_safety tool with multi-ingredient input. Check if normalize_ingredients() is splitting correctly when called through the agent vs directly.

  Query 4: Agent did not call any tools for 'Is benzoyl peroxide good for oily skin?' — violated the system prompt rule to always look up ingredients first
    Root cause: Agent decided it could answer from general knowledge without using the database. System prompt rule was not strong enough.
    Fix: Strengthen system prompt: 'NEVER answer ingredient safety questions without calling lookup_ingredient_safety first, even if you think you know the answer.'

  Query 8: Agen

## 7. Summary stats

In [29]:
successful = [r for r in results if r["success"]]
failed = [r for r in results if not r["success"]]

print("=" * 60)
print("AGENT EVALUATION SUMMARY")
print("=" * 60)
print(f"Total queries:       {len(results)}")
print(f"Successful:          {len(successful)}")
print(f"Failed:              {len(failed)}")

if successful:
    avg_latency = sum(r["latency"] for r in successful) / len(successful)
    print(f"Avg latency:         {avg_latency:.2f}s")

    tool_accuracy = sum(1 for r in results if check_tool_accuracy(r["query"]["expected_tools"], r["tool_calls"]))
    print(f"Tool call accuracy:  {tool_accuracy}/{len(results)} ({tool_accuracy/len(results)*100:.0f}%)")

    # Per-category accuracy
    print("\nPer-category accuracy:")
    for cat in ["ingredient_lookup", "product_query", "combined", "summary", "edge_case"]:
        cat_results = [r for r in results if r["query"]["category"] == cat]
        if cat_results:
            cat_correct = sum(1 for r in cat_results if check_tool_accuracy(r["query"]["expected_tools"], r["tool_calls"]))
            print(f"  {cat:25s}  {cat_correct}/{len(cat_results)} ({cat_correct/len(cat_results)*100:.0f}%)")

if failed:
    print(f"\nFailed queries:")
    for r in failed:
        print(f"  [{r['query']['id']}] {r['query']['query'][:50]}  Error: {r['error'][:80]}")

print(f"\nFailure modes documented: {len(failure_modes)}")
print("=" * 60)

AGENT EVALUATION SUMMARY
Total queries:       20
Successful:          20
Failed:              0
Avg latency:         3.71s
Tool call accuracy:  18/20 (90%)

Per-category accuracy:
  ingredient_lookup          4/5 (80%)
  product_query              4/4 (100%)
  combined                   4/4 (100%)
  summary                    2/3 (67%)
  edge_case                  4/4 (100%)

Failure modes documented: 5


## 8. Cleanup

In [ ]:
# Uncomment to clean up test data
# from app.models.chat import ChatMessage
# db.query(ChatMessage).filter(ChatMessage.user_id == test_user.id).delete()
# db.query(Product).filter(Product.user_id == test_user.id).delete()
# db.query(SkinProfile).filter(SkinProfile.user_id == test_user.id).delete()
# db.query(User).filter(User.id == test_user.id).delete()
# db.commit()
# print("Test data cleaned up")

db.close()
print("Database session closed")